# Identity and Subsample Transforms

This tutorial applies `IdentityPassThrough` and `SubsampleTransform` to `NamedTransformInput` data. Both transforms use `NoFitPerSegmentMixin`: they are **stateless** and require no fitting.

## transform_data Flow

```mermaid
flowchart LR
    A[NamedTransformInput] --> B[transform_data]
    B --> C[metadata]
    B --> D[Output]
    C --> B
```

At the transform level, `transform_data(data, metadata)` receives a single `NamedTransformInput` and returns the transformed result. The pipeline handles `apply_to` / `assign_to` and multi-unit chunking; here we call transforms directly on in-memory data.

## NoFitPerSegmentMixin

Both `IdentityPassThrough` and `SubsampleTransform` inherit `NoFitPerSegmentMixin`:

- **No fit** — No state is learned; `fit_data` is a no-op.
- **Per segment** — When used in the pipeline, each unit/chunk is transformed independently.

## apply_to / assign_to Mental Model

When transforms run inside the pipeline:

- **`apply_to`** — Which data keys to read from (e.g. `features`).
- **`assign_to`** — Where to write results (defaults to `apply_to` if omitted).

In this tutorial we bypass the pipeline and pass `NamedTransformInput` directly, so we use the keys we define (e.g. `features`).

In [ ]:
import numpy as np
from picid.data.data_objects import NamedTransformInput
from picid.transforms.base_transforms.identity import IdentityPassThrough

data = NamedTransformInput(features=np.arange(120).reshape(120, 1).astype(np.float32))
identity = IdentityPassThrough()
out1 = identity.transform_data(data, {"mode": "train"})
print(
    f"Identity: input shape {data['features'].shape} -> output shape {out1['features'].shape}"
)
assert out1["features"].shape == data["features"].shape

In [ ]:
from picid.transforms.base_transforms.subsample import SubsampleTransform

subsample = SubsampleTransform(step=4)
out2 = subsample.transform_data(data.copy(), {"mode": "train"})
print(f"Subsample: 120 elements -> {len(out2['features'])} elements (step=4)")
assert len(out2["features"]) == 30

## Module Focus

This tutorial focuses on **transforms only** — in-memory application of `transform_data` on `NamedTransformInput`. No pipeline, no Datamodule, no DataLoader. For full pipeline integration, see the transforms config and developer guides.